In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv

# 1. Node Embedding Module
class NodeEmbedding(nn.Module):
    def __init__(self, in_channels, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(in_channels, hidden_dim)

    def forward(self, x):
        return self.linear(x)

# 2. Graph Transformer Encoder Module
class GraphTransformerEncoder(nn.Module):
    def __init__(self, hidden_dim, heads=4):
        super().__init__()
        self.layer1 = TransformerConv(hidden_dim, hidden_dim // heads, heads=heads)
        self.layer2 = TransformerConv(hidden_dim, hidden_dim // heads, heads=heads)

    def forward(self, x, edge_index):
        x = self.layer1(x, edge_index)
        x = F.relu(x)
        x = self.layer2(x, edge_index)
        return x

# 3. New Node Encoder Module
class NewNodeEncoder(nn.Module):
    def __init__(self, in_channels, hidden_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_channels, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, x_new):
        return self.encoder(x_new)

# 4. Edge Predictor Module
class EdgePredictor(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_existing, x_new_encoded):
        x_new_expanded = x_new_encoded.expand(x_existing.size(0), -1)
        edge_input = torch.cat([x_existing, x_new_expanded], dim=1)
        edge_logits = self.mlp(edge_input).squeeze()
        return torch.sigmoid(edge_logits)

# 5. Main Graph Transformer Edge Predictor Model
class GraphTransformerEdgePredictor(nn.Module):
    def __init__(self, in_channels, hidden_dim, heads=4):
        super().__init__()
        self.node_embedding = NodeEmbedding(in_channels, hidden_dim)
        self.graph_encoder = GraphTransformerEncoder(hidden_dim, heads)
        self.new_node_encoder = NewNodeEncoder(in_channels, hidden_dim)
        self.edge_predictor = EdgePredictor(hidden_dim)

    def forward(self, x, edge_index, x_new):
        x = self.node_embedding(x)
        x = self.graph_encoder(x, edge_index)
        h_new = self.new_node_encoder(x_new)
        return self.edge_predictor(x, h_new)

In [9]:
from torch_geometric.data import Data
from torch_geometric.utils import dense_to_sparse

In [ ]:
def adjacency_to_edge_index(adj):
    edge_index = adj.nonzero(as_tuple=False).t()
    edge_index = edge_index.numpy()
    edge_set = set(tuple(sorted((i, j))) for i, j in zip(*edge_index))
    edge_index = torch.tensor(list(zip(*edge_set)), dtype=torch.long)
    return edge_index

tensor([[0, 1],
        [1, 2]])


In [ ]:
from torch.utils.data import Dataset, DataLoader

class CircuitGraphDataset(Dataset):
    def __init__(self, graphs):
        """
        Args:
            graphs: List of tuples (x, adj, x_new, target_adj)
                - x: Node features of the graph (Tensor of shape [num_nodes, num_features])
                - adj: Adjacency matrix of the graph (Tensor of shape [num_nodes, num_nodes])
                - x_new: Features of the new node to be added (Tensor of shape [1, num_features])
                - target_adj: Target adjacency matrix after adding the new node (Tensor of shape [num_nodes+1, num_nodes+1])
        """
        self.graphs = graphs

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        x, adj, x_new, target_adj = self.graphs[idx]
        return x, adj, x_new, target_adj

# Example dataset (replace with your actual data)
graphs = [
    (
        torch.tensor([[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 1, 1, 0]], dtype=torch.float),
        torch.tensor([[0, 1, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0]], dtype=torch.float),
        torch.tensor([[0, 0, 0, 0, 1]], dtype=torch.float),
        torch.tensor([[0, 1, 0, 0, 1], [1, 0, 1, 0, 0], [0, 1, 0, 1, 0], [0, 0, 1, 0, 0], [1, 0, 0, 0, 0]], dtype=torch.float),
    )
]

dataset = CircuitGraphDataset(graphs)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [13]:
def train_model(model, data_loader, optimizer, criterion, epochs=10, device='cpu'):
    model.to(device)
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for batch in data_loader:
            # Unpack batch data
            x, adj, x_new, target_adj = batch
            x, adj, x_new, target_adj = x[0].to(device), adj[0].to(device), x_new[0].to(device), target_adj[0].to(device)

            # Convert adjacency matrix to edge_index
            edge_index = dense_to_sparse(adj)[0]

            # Forward pass
            optimizer.zero_grad()
            edge_probs = model(x, edge_index, x_new)

            # Compute loss
            target_edges = target_adj[-1, :-1]  # The new row of the adjacency matrix
            loss = criterion(edge_probs, target_edges)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss:.4f}")

# Initialize model, optimizer, and loss function
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GraphTransformerEdgePredictor(in_channels=5, hidden_dim=32, heads=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for edge probabilities

# Train the model
train_model(model, data_loader, optimizer, criterion, epochs=20, device=device)

Epoch 1/20, Loss: 0.6939
Epoch 2/20, Loss: 0.6821
Epoch 3/20, Loss: 0.6708
Epoch 4/20, Loss: 0.6598
Epoch 5/20, Loss: 0.6486
Epoch 6/20, Loss: 0.6388
Epoch 7/20, Loss: 0.6293
Epoch 8/20, Loss: 0.6197
Epoch 9/20, Loss: 0.6101
Epoch 10/20, Loss: 0.6002
Epoch 11/20, Loss: 0.5901
Epoch 12/20, Loss: 0.5795
Epoch 13/20, Loss: 0.5687
Epoch 14/20, Loss: 0.5575
Epoch 15/20, Loss: 0.5460
Epoch 16/20, Loss: 0.5344
Epoch 17/20, Loss: 0.5226
Epoch 18/20, Loss: 0.5111
Epoch 19/20, Loss: 0.4996
Epoch 20/20, Loss: 0.4883


In [14]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")


Number of parameters in the model: 12001


In [11]:
# Sample node features for 4 components
x = torch.tensor([
    [1, 0, 0, 0, 0],  # Resistor
    [0, 1, 0, 0, 0],  # Capacitor
    [0, 0, 1, 0, 0],  # Inductor
    [0, 0, 1, 1, 0],  # Op-Amp
], dtype=torch.float)

# Adjacency matrix (manually created)
adj = torch.tensor([
    [0, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0]
], dtype=torch.float)

# Convert adjacency matrix to PyG edge_index
edge_index =  dense_to_sparse(adj)[0]  # shape [2, num_edges]

# New component (e.g., a Diode)
x_new = torch.tensor([[0, 0, 0, 0, 1]], dtype=torch.float)  # shape [1, 5]

model = GraphTransformerEdgePredictor(in_channels=5, hidden_dim=32, heads=4)
edge_probs = model(x, edge_index, x_new)

print("Edge probabilities (new node → existing nodes):")
print(edge_probs)



Edge probabilities (new node → existing nodes):
tensor([0.5204, 0.5164, 0.5197, 0.5148], grad_fn=<SigmoidBackward0>)
